<a href="https://colab.research.google.com/github/simecek/dspracticum2026/blob/main/lesson05/03_bert_and_modernbert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 05 · Notebook 3: BERT and ModernBERT

GPT reads a text from left to right and predicts the next token. That is what you need for **writing** text. But many tasks are about **understanding** a text you already have: is this review positive? Which documents answer my question? Which words are names of people? For these, the go-to model is **BERT** (Google, 2018): the same transformer block, but every token sees the **whole** text, to the left and to the right.

For years, BERT and its variants were the workhorse of language processing; Google Search started using it in 2019. In December 2024 came **ModernBERT**: BERT rebuilt with what people learned from LLMs since 2018. And in 2025 **mmBERT**, the same architecture trained on 1,800 languages, including Czech.

**What you will do:**
- see that BERT is GPT with a different attention mask
- let BERT fill in missing words, as in its training
- look inside BERT, and see that the same word gets different vectors in different sentences
- compare BERT with ModernBERT, and try mmBERT on Czech
- *(optional)* fine-tune mmBERT to tell positive Czech movie reviews from negative ones

**Contents:** 0. Setup · 1. GPT vs BERT: a different mask · 2. Fill in the blank · 3. Inside BERT · 4. One word, many vectors · 5. ModernBERT · 6. mmBERT: a ModernBERT for Czech · 7. Fine-tuning a classifier (optional) · 8. Summary

**Before you start:** *Runtime → Change runtime type → T4 GPU* (needed only for section 7).

---
## 0. Setup

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from transformers import pipeline, AutoTokenizer, AutoModel, AutoConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def count(module):
    return sum(p.numel() for p in module.parameters())

def show_predictions(fill, sentence, k=5):
    # Print the k most likely words for the [MASK] in the sentence.
    sentence = sentence.replace("[MASK]", fill.tokenizer.mask_token)    # every model has its own mask token
    print(sentence)
    for r in fill(sentence, top_k=k):
        print(f"   {r['score']:6.1%}  {r['token_str']!r}")

---
## 1. GPT vs BERT: a different mask

Remember the **causal mask** from last week: in GPT, a token may only look at itself and the tokens **before** it, because it must predict what comes next. (Exercise 3 last week asked what happens without it.)

BERT has **no causal mask**: every token looks at every other token. That is the only difference inside the transformer block. GPT is a **decoder** (it generates), BERT is an **encoder** (it turns a text into vectors).

In [ ]:
tokens = ["The", "dog", "barked", "at", "the", "mailman"]
T = len(tokens)
masks = {"GPT (causal mask)": torch.tril(torch.ones(T, T)), "BERT (no mask)": torch.ones(T, T)}

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (title, mask) in zip(axes, masks.items()):
    ax.imshow(mask, cmap="Blues", vmin=0, vmax=1.5)
    ax.set_xticks(range(T), tokens, rotation=90)
    ax.set_yticks(range(T), tokens)
    ax.set_xlabel("... can look at this token")
    ax.set_title(title)
axes[0].set_ylabel("this token ...")
plt.tight_layout()
plt.show()

Take the sentence *"The ___ barked at the mailman."* GPT, reading from the left, sees only *"The"* when it has to fill the blank. BERT sees *"barked"* and *"mailman"* too.

The price: without the mask, BERT cannot generate text token by token. It is not a chatbot.

---
## 2. Fill in the blank

How do you train a model that sees the whole text? "Predict the next token" would be cheating: the next token is right there. Instead, BERT is trained on **masked language modelling**: take a text, hide 15% of its tokens behind a special `[MASK]` token, and let the model guess them. Like GPT, it needs no labels, just a lot of text: BERT read all of English Wikipedia and 11,000 books (3.3 billion words).

The `fill-mask` pipeline lets us play this game with the trained model:

In [ ]:
bert_fill = pipeline("fill-mask", model="google-bert/bert-base-uncased", device=device)

show_predictions(bert_fill, "Paris is the [MASK] of France.")
show_predictions(bert_fill, "The [MASK] barked at the mailman.")
show_predictions(bert_fill, "I went to the [MASK] to buy some bread.")

*(The table about `UNEXPECTED` keys: BERT was also trained on a second task, "is sentence B the continuation of sentence A?", and its weights for that task are in the file, but the `fill-mask` pipeline does not need them. You can ignore it.)*

BERT is not even sure whether it was a man or a dog who barked. It is from 2018; in section 5 we try a newer model on the same sentences.

**Try it:** Your own sentences, e.g. *"The capital of the Czech Republic is [MASK]."*

**Look inside: what did BERT learn about people?**

In [ ]:
show_predictions(bert_fill, "The nurse said that [MASK] would be late.")
show_predictions(bert_fill, "The doctor said that [MASK] would be late.")

The model learned the statistics of its training texts, including stereotypes. A model trained on such texts and used, e.g., to sort job applications can pass the bias on. Try other professions.

---
## 3. Inside BERT

First the tokenizer. BERT uses **WordPiece**, a cousin of BPE: pieces that continue a word start with `##`. This BERT is *uncased*: it lowercases everything. And it adds two special tokens: `[CLS]` at the start and `[SEP]` at the end.

In [ ]:
bert_tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

for text in ["Paris is the capital of France.", "Unbelievably, the tokenizer handles it.", "Příliš žluťoučký kůň"]:
    ids = bert_tokenizer(text)["input_ids"]
    print(bert_tokenizer.convert_ids_to_tokens(ids))

The Czech sentence lost all its accents ("pri ##lis z ##lu ##tou ##cky kun"): this BERT only knows English.

Now the model. `AutoModel` loads the network **without** a task head: it returns one vector per token, not word predictions.

In [ ]:
bert = AutoModel.from_pretrained("google-bert/bert-base-uncased").to(device)
print(f"parameters: {count(bert):,}")
bert

Compare with GPT-2 small from last week:

| | GPT-2 small | BERT-base |
|---|---|---|
| year | 2019 | 2018 |
| type | decoder: predicts the next token | encoder: fills in masked tokens |
| attention mask | causal | none, everyone sees everyone |
| blocks × vector size × heads | 12 × 768 × 12 | 12 × 768 × 12 |
| queries, keys, values | one layer `c_attn` | three layers `query`, `key`, `value` |
| vocabulary | 50,257 (BPE) | 30,522 (WordPiece) |
| context | 1,024 tokens | 512 tokens |
| parameters | 124 million | 110 million |

Almost twins! A few details differ: BERT has one more embedding table (`token_type_embeddings`, which marks sentence A and sentence B), and its LayerNorm comes *after* the attention and the MLP, not before.

---
## 4. One word, many vectors

BERT turns every token into a vector, after all 12 blocks. Each of these vectors has seen the whole sentence, so the **same word gets a different vector in a different sentence**. (Last week's embedding table `wte` gave each token one fixed vector; this is what the blocks add.)

Let's take the word *"bank"* in four sentences, two about rivers and two about money:

In [ ]:
sentences = [
    "I sat on the bank of the river.",
    "The river bank was covered with grass.",
    "I deposited money at the bank.",
    "The bank approved my loan.",
]

bank_vectors = []
for s in sentences:
    enc = bert_tokenizer(s, return_tensors="pt").to(device)
    with torch.no_grad():
        out = bert(**enc).last_hidden_state[0]           # (tokens, 768): one vector per token
    position = bert_tokenizer.convert_ids_to_tokens(enc["input_ids"][0]).index("bank")
    bank_vectors.append(out[position])

V = F.normalize(torch.stack(bank_vectors), dim=-1)
similarity = (V @ V.T).cpu()                             # cosine similarity of all pairs

plt.imshow(similarity, cmap="viridis", vmin=0, vmax=1)
plt.xticks(range(4), [f"sentence {i}" for i in range(4)])
plt.yticks(range(4), sentences)
for i in range(4):
    for j in range(4):
        plt.text(j, i, f"{similarity[i, j]:.2f}", ha="center", va="center", color="white")
plt.colorbar()
plt.title('similarity of the vectors for "bank"')
plt.show()

The river banks are similar to each other, the money banks too, and the two groups are much less similar. BERT tells the two meanings apart, from the context.

**What is this good for?** With such vectors you can:
- **classify** texts: add a small layer on top of the vectors and train it on labelled examples (section 7)
- **search** by meaning: turn documents and questions into vectors and look for the closest ones (*embedding models*, e.g. for RAG, are BERT-like encoders)
- **find names, places, dates** in a text: classify every token

---
## 5. ModernBERT

Six years after BERT, [ModernBERT](https://huggingface.co/blog/modernbert) (Answer.AI and LightOn, December 2024) took the BERT recipe and updated it with what people had learned from building LLMs:

| | BERT-base (2018) | ModernBERT-base (2024) |
|---|---|---|
| training data | 3.3 billion words: Wikipedia and books | 2 trillion tokens: web, code, science |
| context | 512 tokens | 8,192 tokens |
| positions | learned table (like GPT-2's `wpe`) | **RoPE**: rotates queries and keys by an angle that depends on the position |
| attention | every block: all tokens | every 3rd block: all tokens ("global"); the others: only the 128 nearest tokens ("local") |
| MLP | GELU | **GeGLU**: one half of the MLP decides how much of the other half to let through |
| biases | in every layer | almost none |
| tokenizer | WordPiece, 30,522, lowercase | BPE, 50,368, keeps case, knows code |
| blocks, parameters | 12, 110 million | 22, 150 million |

Local attention and dropping the padding tokens make ModernBERT fast, even on long texts. Let's play the same game:

In [ ]:
modern_fill = pipeline("fill-mask", model="answerdotai/ModernBERT-base", device=device)

show_predictions(modern_fill, "Paris is the [MASK] of France.")
show_predictions(modern_fill, "The [MASK] barked at the mailman.")
show_predictions(modern_fill, "I went to the [MASK] to buy some bread.")
show_predictions(modern_fill, "def add(a, b):\n    return a [MASK] b")

The dog is back! And thanks to the code in its training data, ModernBERT also reads Python.

Notice the space at the start of the predicted tokens: ModernBERT uses a GPT-style BPE tokenizer.

**Look inside: local and global attention.** The config lists the type of attention in every block:

In [ ]:
modern_config = AutoConfig.from_pretrained("answerdotai/ModernBERT-base")
print("blocks:", modern_config.num_hidden_layers, "  context:", modern_config.max_position_embeddings,
      "  local window:", modern_config.local_attention, "tokens")
for i, kind in enumerate(modern_config.layer_types):
    print(f"block {i:2}: {kind}")

Every 3rd block sees the whole text (`full_attention`), the others only a window of 128 tokens (`sliding_attention`). Information still travels across the whole text, just not in every block. For a text of 8,192 tokens, a full attention block computes 8,192 × 8,192 ≈ 67 million scores per head, a local one only 8,192 × 128 ≈ 1 million.

---
## 6. mmBERT: a ModernBERT for Czech

ModernBERT read English (and code). In September 2025, Johns Hopkins University released [mmBERT](https://huggingface.co/jhu-clsp/mmBERT-base): the ModernBERT architecture trained on 3 trillion tokens in more than 1,800 languages.

In [ ]:
mm_fill = pipeline("fill-mask", model="jhu-clsp/mmBERT-base", device=device)

show_predictions(mm_fill, "Praha je hlavní [MASK] České republiky.")
show_predictions(mm_fill, "Paris is the [MASK] of France.")
show_predictions(mm_fill, "Šel jsem do [MASK] koupit chleba.")

**Look inside: where are the parameters?** To cover 1,800 languages, mmBERT uses the tokenizer of Google's Gemma, with 256,000 tokens. Compare the parameters of the embedding table with the rest of the model:

In [ ]:
for name, fill in [("ModernBERT-base", modern_fill), ("mmBERT-base", mm_fill)]:
    model = fill.model
    emb = model.get_input_embeddings()
    print(f"{name:16} vocabulary {emb.num_embeddings:>7,}   embedding table {count(emb):>12,}   all parameters {count(model):>12,}")

print(mm_fill.tokenizer.tokenize("Příliš žluťoučký kůň úpěl ďábelské ódy."))

In mmBERT, about two thirds of all parameters are the embedding table: a vector for each of the 256,000 tokens. In return, Czech words are cut into sensible pieces.

**Try it:** Fill-mask in another language you know. Is there one that mmBERT does not know?

---
## 7. Fine-tuning a classifier (optional)

Now we teach a BERT a new task: does a ČSFD review (notebook 2) like the movie or not? We take the reviews with 0-1 stars as **negative** and 4-5 stars as **positive**, and drop the lukewarm ones.

This is **fine-tuning**, just as with the image models in lessons 2 and 3: a pre-trained network, plus a new small layer (the *head*) that we train on labelled examples. We use `mmBERT-small` (140 million parameters), so that it trains in a few minutes on the T4 GPU.

In [ ]:
from datasets import load_dataset

reviews = load_dataset("fewshot-goes-multilingual/cs_csfd-movie-reviews")

def positive_or_negative(split, n):
    ds = reviews[split].filter(lambda row: row["rating_int"] in (0, 1, 4, 5))
    ds = ds.map(lambda row: {"label": int(row["rating_int"] >= 4)})     # 1 = positive, 0 = negative
    return ds.shuffle(seed=42).select(range(n))

train_ds = positive_or_negative("train", 2000)
test_ds = positive_or_negative("test", 500)
print(train_ds)
print("positive in training data:", sum(train_ds["label"]), "of", len(train_ds))

The texts must be tokenized first. `map` with `batched=True` gives the function 1,000 rows at once, which is much faster. Reviews longer than 256 tokens are cut off (`truncation`).

In [ ]:
from transformers import AutoModelForSequenceClassification

model_name = "jhu-clsp/mmBERT-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["comment"], truncation=True, max_length=256)

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2, id2label={0: "negative", 1: "positive"}, label2id={"negative": 0, "positive": 1})
print(model.classifier)

The load report says it: the weights for word prediction (`decoder`) are `UNEXPECTED`, because we do not need them now, and the `classifier` is `MISSING`: it is new, with random weights. The classifier takes the **average** of all token vectors (384 numbers) and turns it into 2 scores, one per class.

We could write the training loop ourselves, as last week. But `transformers` has one ready: the **`Trainer`**. It does the same 5 steps, plus batching, padding, evaluation, logging and the GPU:

In [ ]:
from transformers import Trainer, TrainingArguments

def accuracy(eval_pred):
    logits, labels = eval_pred
    return {"accuracy": (logits.argmax(-1) == labels).mean()}

args = TrainingArguments(
    output_dir="csfd-classifier",
    num_train_epochs=2,
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",              # measure the accuracy after every epoch
    logging_steps=25,
    save_strategy="no",                 # do not save checkpoints during training
    fp16=torch.cuda.is_available(),     # faster on the GPU
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds,
                  processing_class=tokenizer, compute_metrics=accuracy)
trainer.train()

In [ ]:
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=device)

my_reviews = [
    "Naprosto skvělý film, herci výborní, doporučuji!",
    "Nuda, nuda, šeď. Škoda času.",
    "Ze začátku jsem se nudil, ale konec to zachránil.",
]
for review, result in zip(my_reviews, classifier(my_reviews)):
    print(f"{result['label']:8} {result['score']:.0%}   {review}")

**Try it:**
- Write your own reviews. Can you fool the classifier, e.g. with irony?
- Use all 5 classes (0-5 stars, `num_labels=6`) instead of positive/negative. How good is it at guessing the exact number of stars?
- Train `answerdotai/ModernBERT-base` on the same data. It does not know Czech: how much worse is it?
- Upload your classifier: `trainer.push_to_hub()` or `model.push_to_hub(...)` and `tokenizer.push_to_hub(...)`, as in notebooks 1 and 2.

---
## 8. Summary

| | GPT | BERT |
|---|---|---|
| attention | causal: only the tokens before | all tokens |
| trained on | predicting the next token | filling in masked tokens |
| good for | generating text | understanding text: classification, search, finding names |
| pipeline | `text-generation` | `fill-mask`, `text-classification`, `feature-extraction` |
| model classes | `AutoModelForCausalLM` | `AutoModel`, `AutoModelForMaskedLM`, `AutoModelForSequenceClassification` |

**New words:** encoder, decoder, masked language modelling, `[MASK]` / `[CLS]` / `[SEP]`, WordPiece, contextual embeddings, RoPE, local and global attention, GeGLU, classification head, fine-tuning, `Trainer`.

### Exercises
1. Find a sentence where BERT and ModernBERT disagree, and decide who is right.
2. Repeat the "bank" experiment of section 4 with a Czech word with two meanings (e.g. *"zámek"*: a castle and a lock) and mmBERT (`AutoModel.from_pretrained("jhu-clsp/mmBERT-base")`). Careful: the word may be split into several tokens.
3. In section 7, train with 200, 500 and 2,000 reviews. How does the accuracy depend on the amount of labelled data?

### Further reading
- [BERT paper](https://arxiv.org/abs/1810.04805), Jay Alammar: [The Illustrated BERT](https://jalammar.github.io/illustrated-bert/)
- [Finally, a replacement for BERT](https://huggingface.co/blog/modernbert): the ModernBERT blog post, and the [paper](https://arxiv.org/abs/2412.13663)
- [mmBERT](https://huggingface.co/blog/mmbert): the multilingual version